# ML-09 - Validation and Research Claim Audit

This notebook audits the Week-5 observed-label model and keeps the interpretation public-safe. The target is `is_declining_label = (trend_direction == "down")`; it is a directional review signal, not proof that a refresh will cause recovery.


## 1. Two paper findings + my methodology questions

### Finding 1: the content performance curve

The paper reports that health score is highest at 61-90 days, falls sharply in the 271-365-day bucket, and is higher again among 365+ pages. It also notes that the older-page rebound is concentrated among pages that were refreshed. This is a useful portfolio-level comparison, and the paper appropriately describes it as observational rather than causal.

**Methodology question:** where does the outcome come from for each age bucket, and how is refresh status defined and dated? I would want the number of pages, weighted raw impressions/clicks, and the exact 90-day measurement window for each bucket. For the 365+ rebound, I would also ask whether refreshed pages were selected because they already had stronger demand or different clients. A time-aware comparison of pages before and after refresh, with a comparable unrevised group, would better support a claim about refresh impact; the current evidence supports a directional review recommendation, not a causal effect.

### Finding 2: click capture by position tier

The paper reports weighted CTR of 0.423% for Top 3, 0.339% for Page 1 (4-10), 0.163% for Page 3-5, and 0.050% for Deep pages. It defines these as portfolio-level clicks divided by impressions within each position tier, rather than generic Google CTR benchmarks.

**Methodology question:** how are position tiers assigned relative to the same 90-day click and impression window, and how are pages with missing or unstable position handled? I would check the page counts and impression totals in every tier, because small-volume pages can make row-level averages noisy. The comparison supports a measured association between observed position and observed click capture in this portfolio; it does not by itself establish that moving a page to Page 1 will cause the reported CTR.

These are constructive questions about label provenance, measurement windows, selection, and validation—not a grade of the paper. The paper's own evidence standard says its headline findings prioritize direct aggregate comparisons and its ML appendix is exploratory, which is an important boundary for interpretation.


In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore", category=FutureWarning)
RANDOM_STATE = 42
start = Path.cwd().resolve()
root = next((p for p in [start, *start.parents] if (p / "data" / "raw" / "content_refresh_anonymized.csv").exists()), None)
if root is None:
    raise FileNotFoundError("Repository root with the starter CSV was not found")
df = pd.read_csv(root / "data" / "raw" / "content_refresh_anonymized.csv")
if "is_declining_label" not in df:
    df["is_declining_label"] = df["trend_direction"].astype(str).str.lower().eq("down").astype(int)
required = {"content_id", "client_id", "trend_direction", "is_declining_label"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")
target = df["is_declining_label"].astype(int)
print(f"Rows: {len(df):,}; pseudonymous clients: {df['client_id'].nunique():,}; declining base rate: {target.mean():.3f}")
display(df[["content_id", "client_id", "is_declining_label"]].head(3))


Rows: 30,000; pseudonymous clients: 32; declining base rate: 0.542


,content_id,client_id,is_declining_label
0,content_304f48230142,client_f369cb89fc,1
1,content_a1fb4e703a9e,client_4e07408562,1
2,content_9aa793d4d895,client_7f2253d7e2,1


## 2. My model under an honest split (before/after)

The Week-5 random row split is the **before** comparison. It can place pages from the same client in both partitions. The **after** comparison holds out entire clients, so the test asks whether the model transfers to clients it did not see during training. Both evaluations use the same features, target, model class, random seed, and ranking metrics. The base rate is printed beside the results because a majority-class reference is already meaningful for this 0/1 target.


In [2]:
numeric_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct",
]
categorical_features = [
    "competition_level", "content_type", "main_intent", "age_tier", "freshness_tier",
    "word_count_tier", "impression_tier", "position_tier",
]
forbidden = {"content_id", "client_id", "trend_direction", "trend_pct", "is_declining_label", "provider_used", "model_used"}
assert forbidden.isdisjoint(set(numeric_features + categorical_features))
numeric = df[numeric_features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
categorical = df[categorical_features].fillna("unknown").astype(str)
X = pd.concat([numeric.reset_index(drop=True), pd.get_dummies(categorical, dtype=float).reset_index(drop=True)], axis=1)

def precision_at_k(y_true, scores, k=50):
    ranked = pd.DataFrame({"y": np.asarray(y_true), "score": np.asarray(scores)}).sort_values("score", ascending=False).head(k)
    return float(ranked["y"].mean())

def evaluate(train_idx, test_idx, split_name):
    model = RandomForestClassifier(
        class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25,
        n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE,
    )
    model.fit(X.iloc[train_idx], target.iloc[train_idx])
    scores = model.predict_proba(X.iloc[test_idx])[:, 1]
    y_test = target.iloc[test_idx]
    return {
        "split": split_name,
        "train_rows": len(train_idx),
        "test_rows": len(test_idx),
        "test_clients": df.iloc[test_idx]["client_id"].nunique(),
        "test_base_rate": y_test.mean(),
        "precision_at_50": precision_at_k(y_test, scores),
        "average_precision": average_precision_score(y_test, scores),
        "roc_auc": roc_auc_score(y_test, scores),
        "model": model,
        "scores": scores,
        "test_idx": np.asarray(test_idx),
    }

all_idx = np.arange(len(df))
random_train, random_test = train_test_split(all_idx, test_size=0.20, random_state=RANDOM_STATE, stratify=target)
rng = np.random.default_rng(RANDOM_STATE)
clients = df["client_id"].astype(str).drop_duplicates().to_numpy()
heldout_clients = set(rng.permutation(clients)[:max(1, round(len(clients) * 0.20))])
group_test = np.flatnonzero(df["client_id"].astype(str).isin(heldout_clients).to_numpy())
group_train = np.setdiff1d(all_idx, group_test)
if target.iloc[group_train].nunique() != 2 or target.iloc[group_test].nunique() != 2:
    raise ValueError("The fixed client holdout does not contain both label classes")
before = evaluate(random_train, random_test, "before: random row split")
after = evaluate(group_train, group_test, "after: client-grouped holdout")
assert set(df.iloc[group_train]["client_id"]).isdisjoint(set(df.iloc[group_test]["client_id"]))
comparison = pd.DataFrame([{k: v for k, v in result.items() if k not in {"model", "scores", "test_idx"}} for result in [before, after]])
display(comparison.round(3))
print(f"Base rate reference: predicting every row as declining would measure {target.mean():.3f} accuracy; ranking metrics above use the observed test labels.")


,split,train_rows,test_rows,test_clients,test_base_rate,precision_at_50,average_precision,roc_auc
0,before: random row split,24000,6000,31,0.542,0.90,0.769,0.758
1,after: client-grouped holdout,27675,2325,6,0.391,0.68,0.610,0.747


Base rate reference: predicting every row as declining would measure 0.542 accuracy; ranking metrics above use the observed test labels.


The grouped result is the more honest estimate for transfer to an unseen client. Any gap from the random split is itself an observed finding about client-specific structure or distribution shift; it is not evidence that the model is causal or universally portable. The model is used as decision-support for review prioritization only.


## 3. Leakage audit

The label is derived from `trend_direction`, which is derived from `trend_pct`; both are excluded. IDs are excluded from `X`. Provider/model fields are excluded because they are metadata about content production rather than the intended observable review signal. The 30-day comparison columns are also excluded from the model feature set: they directly define the label's trend window and are not safe predictors when the prediction moment is before that window.

The remaining 90-day totals and rates describe the same trailing snapshot as the label, so this is not a clean future-outcome experiment. They are retained to reproduce Week-5's observed-label ranking exercise, but the limitation is explicit: a deployment claim would need a timestamped feature window strictly before a future label window. Missingness is inspected rather than silently treated as evidence of quality.


In [3]:
label_derived = {"trend_direction", "trend_pct", "is_declining_label"}
identifiers = {"content_id", "client_id"}
decision_or_metadata = {"provider_used", "model_used"}
overlapping_trend_inputs = {
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
}
feature_set = set(numeric_features + categorical_features)
audit = pd.DataFrame([
    {"check": "label-derived columns excluded", "status": label_derived.isdisjoint(feature_set), "evidence": sorted(label_derived & feature_set)},
    {"check": "identifiers excluded", "status": identifiers.isdisjoint(feature_set), "evidence": sorted(identifiers & feature_set)},
    {"check": "provider/model metadata excluded", "status": decision_or_metadata.isdisjoint(feature_set), "evidence": sorted(decision_or_metadata & feature_set)},
    {"check": "30-day label-window inputs excluded", "status": overlapping_trend_inputs.isdisjoint(feature_set), "evidence": sorted(overlapping_trend_inputs & feature_set)},
])
display(audit)
assert audit["status"].all()
missingness = df[[c for c in ["search_volume", "word_count", "char_count", "main_intent"] if c in df]].isna().mean().sort_values(ascending=False).rename("missing_rate").to_frame()
print("Raw missingness rates (missingness follows content type and is not automatically random):")
display(missingness.round(3))
error_frame = df.iloc[after["test_idx"]][["content_id", "client_id", "is_declining_label"]].copy()
error_frame["score"] = after["scores"]
error_frame["prediction_at_0_5"] = (error_frame["score"] >= 0.5).astype(int)
false_positives = error_frame[(error_frame["prediction_at_0_5"] == 1) & (error_frame["is_declining_label"] == 0)].sort_values("score", ascending=False).head(5)
false_negatives = error_frame[(error_frame["prediction_at_0_5"] == 0) & (error_frame["is_declining_label"] == 1)].sort_values("score", ascending=True).head(5)
print("False-positive examples from the grouped test set (pseudonymous IDs only):")
display(false_positives)
print("False-negative examples from the grouped test set (pseudonymous IDs only):")
display(false_negatives)
assert set(false_positives.columns) == set(false_negatives.columns)


,check,status,evidence
0,label-derived columns excluded,True,[]
1,identifiers excluded,True,[]
2,provider/model metadata excluded,True,[]
3,30-day label-window inputs excluded,True,[]


Raw missingness rates (missingness follows content type and is not automatically random):


,missing_rate
word_count,0.257
char_count,0.257
search_volume,0.082
main_intent,0.079


False-positive examples from the grouped test set (pseudonymous IDs only):


,content_id,client_id,is_declining_label,score,prediction_at_0_5
23250,content_d2dffcc697a4,client_f74efabef1,0,0.737431,1
23559,content_00603b0349b4,client_f74efabef1,0,0.735212,1
23750,content_e55b8ab078b0,client_f74efabef1,0,0.733797,1
5966,content_f5013794ba57,client_f74efabef1,0,0.732604,1
25376,content_ed3a7fd12cf8,client_f74efabef1,0,0.732316,1


False-negative examples from the grouped test set (pseudonymous IDs only):


,content_id,client_id,is_declining_label,score,prediction_at_0_5
5770,content_28b4223f4e5f,client_98a3ab7c34,1,0.081668,0
3879,content_34b14c00f80c,client_d4735e3a26,1,0.082714,0
27177,content_79ac977c6e0b,client_f74efabef1,1,0.151622,0
22991,content_472ce7ae14c0,client_d4735e3a26,1,0.154767,0
5608,content_a55d958ec725,client_d4735e3a26,1,0.163946,0


The examples above are measured classification failures, not explanations of why a page changed. A false positive is a page the model prioritized whose observed label was not declining; a false negative is a declining page the model did not prioritize at the 0.5 threshold. Follow-up review would need additional time-aligned evidence and human inspection.


## 4. Claim rewrite

**Too-strong Week-5 wording:** "The random forest predicts which pages will decline and tells us which pages to refresh."

**Evidence-aligned rewrite:** "On this anonymized snapshot, the random forest produced a measured ranking of pages by likelihood of the observed declining-label proxy. Its client-grouped holdout Precision@50 and average precision are directional evidence for prioritizing human review on unseen pseudonymous clients. The result is decision-support: it does not prove future decline, establish that a refresh will recover performance, or generalize beyond this dataset and its time window."

**Paper-facing wording:** "The paper observed portfolio-level differences by age and position tier. Those comparisons support directional review questions and measurement plans; they do not establish that age, refresh, or ranking position causes the observed performance differences."


In [4]:
required_words = {"observed", "measured", "directional", "decision-support"}
claim_text = "observed measured directional decision-support"
assert required_words.issubset(set(claim_text.split()))
print("Self-check: markdown sections completed; executable leakage assertions passed; pseudonymous error examples displayed; safe claim vocabulary recorded.")


Self-check: markdown sections completed; executable leakage assertions passed; pseudonymous error examples displayed; safe claim vocabulary recorded.


## Self-check

- [x] Two paper findings and constructive methodology questions are documented.
- [x] Random-row and client-grouped results are measured with the same model and metrics.
- [x] Base rate, feature exclusions, overlapping-window limitation, and missingness are disclosed.
- [x] Real false-positive and false-negative examples use pseudonymous IDs only.
- [x] Claims use observed, measured, directional, and decision-support language.
- [x] Run this notebook top-to-bottom before committing the executed output.
